# LeetCode 1184 — Distance Between Bus Stops bằng Python

**Link đề:** https://leetcode.com/problems/distance-between-bus-stops/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def distanceBetweenBusStops(self, distance: List[int], start: int, destination: int) -> int:
        ...
```

## Mục tiêu học
1. Mô hình hóa **mảng vòng tròn** (circular array): đi tiếp từ trạm `n-1` quay về trạm `0` nhờ phép `% n`.
2. Baseline: **mô phỏng xe buýt chạy** theo hai chiều và so sánh.
3. Insight: quãng đường chiều kim đồng hồ là **tổng một đoạn liên tiếp**; chiều ngược = **tổng cả vòng − đoạn đó**.
4. Kỹ thuật **chuẩn hóa** `start <= destination` bằng cách hoán đổi, và **prefix sum** cho nhiều truy vấn.
5. Trace, chứng minh, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Một tuyến xe buýt có `n` trạm đánh số `0..n-1` xếp **thành vòng tròn**. `distance[i]` là khoảng cách giữa trạm `i` và trạm `(i + 1) % n`.

Xe chạy được **cả hai chiều** (thuận và ngược chiều kim đồng hồ). Trả về **khoảng cách ngắn nhất** giữa trạm `start` và `destination`.

`distance = [1, 2, 3, 4]` (vòng: 0 —1— 1 —2— 2 —3— 3 —4— 0):

| start | destination | chiều thuận | chiều ngược | Output |
|---:|---:|---|---|---:|
| 0 | 1 | 1 | 2+3+4 = 9 | `1` |
| 0 | 2 | 1+2 = 3 | 3+4 = 7 | `3` |
| 0 | 3 | 1+2+3 = 6 | 4 | `4` |

**Constraints:**
- `1 <= n <= 10^4`, `distance.length == n`
- `0 <= start, destination < n`
- `0 <= distance[i] <= 10^4`

In [1]:
from typing import List

distance = [1, 2, 3, 4]
n = len(distance)
for i in range(n):
    print(f"trạm {i} --{distance[i]}--> trạm {(i + 1) % n}")
print("chu vi cả vòng =", sum(distance))

trạm 0 --1--> trạm 1
trạm 1 --2--> trạm 2
trạm 2 --3--> trạm 3
trạm 3 --4--> trạm 0
chu vi cả vòng = 10


## 2. Nền tảng Python cần biết

### 2.1. Phép `% n` để "quay vòng"

`(i + 1) % n` cho trạm kế tiếp; khi `i = n - 1` thì `(n - 1 + 1) % n = 0` → quay về đầu. Chiều ngược: `(i - 1) % n` — trong Python, `-1 % 4 = 3` (luôn không âm khi `n > 0`).

In [2]:
n = 4
print([(i + 1) % n for i in range(n)])   # trạm kế tiếp theo chiều thuận
print([(i - 1) % n for i in range(n)])   # trạm kế tiếp theo chiều ngược
print(-1 % 4, -5 % 4)

[1, 2, 3, 0]
[3, 0, 1, 2]
3 3


### 2.2. `sum(a[i:j])` — tổng một đoạn

`sum(distance[0:3])` = `distance[0] + distance[1] + distance[2]`: đúng quãng đường từ trạm 0 tới trạm 3 theo chiều thuận (đi qua 3 cạnh `0-1`, `1-2`, `2-3`).

Quy tắc: đi thuận từ `s` tới `d` (với `s <= d`) = `sum(distance[s:d])`.

In [3]:
distance = [1, 2, 3, 4]
print(sum(distance[0:3]))   # 0→3: 1+2+3
print(sum(distance[1:3]))   # 1→3: 2+3
print(sum(distance[2:2]))   # s == d: đoạn rỗng → 0

6
5
0


### 2.3. Hoán đổi hai biến

`a, b = b, a` đổi giá trị trong một dòng. Dùng để đảm bảo `start <= destination`.

### 2.4. `min(a, b)`
Trả giá trị nhỏ hơn.

In [4]:
start, destination = 3, 1
if start > destination:
    start, destination = destination, start
print(start, destination)
print(min(6, 4))

1 3
4


## 3. Cách 1 — Baseline: mô phỏng xe chạy hai chiều

- Chiều thuận: từ `start`, cộng `distance[cur]` rồi `cur = (cur + 1) % n`, tới khi `cur == destination`.
- Chiều ngược: từ `start`, đi về trạm `prev = (cur - 1) % n`, cộng `distance[prev]` (cạnh nối `prev` và `cur`), tới khi gặp `destination`.

In [5]:
def bus_simulate(distance: List[int], start: int, destination: int) -> int:
    n = len(distance)
    # chiều thuận
    cw, cur = 0, start
    while cur != destination:
        cw += distance[cur]
        cur = (cur + 1) % n
    # chiều ngược
    ccw, cur = 0, start
    while cur != destination:
        prev = (cur - 1) % n
        ccw += distance[prev]
        cur = prev
    return min(cw, ccw)

print(bus_simulate([1, 2, 3, 4], 0, 1), bus_simulate([1, 2, 3, 4], 0, 2), bus_simulate([1, 2, 3, 4], 0, 3))

1 3 4


Khi đi ngược từ `cur` sang `prev`, cạnh đi qua là cạnh `prev → prev+1 = cur`, có độ dài `distance[prev]` (không phải `distance[cur]`). Đây là chỗ dễ nhầm nhất.

### Nhận xét
Cách này đúng và `O(n)`, nhưng đi hai vòng lặp và phải cẩn thận chỉ số. Hai chiều cộng lại **luôn bằng cả vòng**: `cw + ccw = sum(distance)`. Vậy chỉ cần tính **một** chiều!

In [6]:
d = [1, 2, 3, 4]
for s, t in [(0, 1), (0, 2), (1, 3), (3, 0)]:
    n = len(d)
    cw, cur = 0, s
    while cur != t:
        cw += d[cur]; cur = (cur + 1) % n
    print(f"{s}->{t}: thuận = {cw}, cả vòng - thuận = {sum(d) - cw}")

0->1: thuận = 1, cả vòng - thuận = 9
0->2: thuận = 3, cả vòng - thuận = 7
1->3: thuận = 5, cả vòng - thuận = 5
3->0: thuận = 4, cả vòng - thuận = 6


## 4. Insight

1. Hoán đổi để `s = min(start, destination)`, `t = max(...)`. Khoảng cách không phụ thuộc chiều đi nên đổi vai trò hai trạm không ảnh hưởng.
2. Chiều thuận từ `s` tới `t` đi qua các cạnh `s, s+1, ..., t-1` → `clockwise = sum(distance[s:t])` (không cần `% n` vì không vượt qua trạm 0).
3. Chiều ngược = phần còn lại của vòng: `total - clockwise`.
4. Đáp án = `min(clockwise, total - clockwise)`.

## 5. Cách 2 — Lời giải chính: tổng đoạn + phần bù

**Mổ xẻ:**
1. Nếu `start > destination`: hoán đổi.
2. `clockwise = sum(distance[start:destination])`.
3. `total = sum(distance)`.
4. Trả `min(clockwise, total - clockwise)`.

In [7]:
def bus_distance(distance: List[int], start: int, destination: int) -> int:
    if start > destination:
        start, destination = destination, start
    clockwise = sum(distance[start:destination])
    total = sum(distance)
    return min(clockwise, total - clockwise)

print(bus_distance([1, 2, 3, 4], 0, 1), bus_distance([1, 2, 3, 4], 0, 2), bus_distance([1, 2, 3, 4], 0, 3))
print(bus_distance([1, 2, 3, 4], 3, 0))   # hoán đổi: 0 → 3

1 3 4
4


## 6. TRACE từng bước

Ví dụ `distance = [7, 10, 1, 12, 11, 14, 5, 0]`, `start = 7`, `destination = 2`.

| bước | việc | giá trị |
|---|---|---|
| 1 | `start > destination` → hoán đổi | `s = 2`, `t = 7` |
| 2 | cạnh đi thuận | `distance[2..6] = [1, 12, 11, 14, 5]` |
| 3 | `clockwise` | `43` |
| 4 | `total` | `60` |
| 5 | `counter = total - clockwise` | `17` (đi 7→0→1→2: `0 + 7 + 10`) |
| 6 | `min(43, 17)` | **17** |

In [8]:
def trace_bus(distance: List[int], start: int, destination: int) -> int:
    print(f"distance={distance}, start={start}, destination={destination}")
    if start > destination:
        start, destination = destination, start
        print(f"  hoán đổi → s={start}, t={destination}")
    edges = distance[start:destination]
    clockwise = sum(edges)
    total = sum(distance)
    print(f"  cạnh chiều thuận (index {start}..{destination - 1}): {edges} → clockwise = {clockwise}")
    print(f"  total = {total} → chiều ngược = {total - clockwise}")
    ans = min(clockwise, total - clockwise)
    print(f"  đáp án = min({clockwise}, {total - clockwise}) = {ans}")
    return ans

trace_bus([7, 10, 1, 12, 11, 14, 5, 0], 7, 2)
print()
trace_bus([1, 2, 3, 4], 0, 3)

distance=[7, 10, 1, 12, 11, 14, 5, 0], start=7, destination=2
  hoán đổi → s=2, t=7
  cạnh chiều thuận (index 2..6): [1, 12, 11, 14, 5] → clockwise = 43
  total = 60 → chiều ngược = 17
  đáp án = min(43, 17) = 17

distance=[1, 2, 3, 4], start=0, destination=3
  cạnh chiều thuận (index 0..2): [1, 2, 3] → clockwise = 6
  total = 10 → chiều ngược = 4
  đáp án = min(6, 4) = 4


4

## 7. Cách 3 — Một vòng lặp, không cắt list

Duyệt mọi cạnh `i` một lần: cạnh nằm trong `[s, t)` thì cộng vào `clockwise`, cộng mọi cạnh vào `total`. Không tạo list mới bằng slicing → bộ nhớ `O(1)`.

In [9]:
def bus_one_pass(distance: List[int], start: int, destination: int) -> int:
    s, t = min(start, destination), max(start, destination)
    clockwise = total = 0
    for i, d in enumerate(distance):
        total += d
        if s <= i < t:          # so sánh chuỗi: s <= i và i < t
            clockwise += d
    return min(clockwise, total - clockwise)

print(bus_one_pass([1, 2, 3, 4], 0, 3), bus_one_pass([7, 10, 1, 12, 11, 14, 5, 0], 7, 2))

4 17


`s <= i < t` là **so sánh chuỗi** của Python, tương đương `s <= i and i < t`.

## 8. Cách 4 — Prefix sum (nhiều truy vấn)

Nếu phải trả lời **nhiều** cặp `(start, destination)` trên cùng tuyến, tính `pre` một lần: `clockwise = pre[t] - pre[s]` trong `O(1)`.

In [10]:
from itertools import accumulate

class BusRoute:
    def __init__(self, distance: List[int]):
        self.pre = [0] + list(accumulate(distance))
        self.total = self.pre[-1]

    def query(self, start: int, destination: int) -> int:
        s, t = min(start, destination), max(start, destination)
        cw = self.pre[t] - self.pre[s]
        return min(cw, self.total - cw)

route = BusRoute([1, 2, 3, 4])
print(route.pre, [route.query(0, d) for d in range(4)])

[0, 1, 3, 6, 10] [0, 1, 3, 4]


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def distanceBetweenBusStops(self, distance: List[int], start: int, destination: int) -> int:
        if start > destination:
            start, destination = destination, start
        clockwise = sum(distance[start:destination])
        return min(clockwise, sum(distance) - clockwise)

sol = Solution()
print(sol.distanceBetweenBusStops([1, 2, 3, 4], 0, 1))
print(sol.distanceBetweenBusStops([1, 2, 3, 4], 0, 2))
print(sol.distanceBetweenBusStops([1, 2, 3, 4], 0, 3))

1
3
4


In [12]:
class SolutionOnePass:
    def distanceBetweenBusStops(self, distance: List[int], start: int, destination: int) -> int:
        s, t = sorted((start, destination))
        cw = total = 0
        for i, d in enumerate(distance):
            total += d
            if s <= i < t:
                cw += d
        return min(cw, total - cw)

print(SolutionOnePass().distanceBetweenBusStops([1, 2, 3, 4], 3, 0))

4


`sorted((a, b))` trả list đã sắp xếp `[min, max]`, gán đồng thời vào `s, t` — một cách khác để chuẩn hóa.

## 10. Vì sao đúng?

- Trên vòng tròn có đúng **hai** đường đơn giữa hai trạm khác nhau: theo chiều thuận và chiều ngược. Đi vòng nhiều hơn chỉ dài thêm (các khoảng cách `>= 0`), nên đáp án là min của hai đường đó.
- Với `s < t`, đường thuận đi qua các cạnh `s, s+1, ..., t-1` — đúng `distance[s:t]`.
- Hai đường **không chung cạnh nào** và **hợp lại thành cả vòng**, nên đường ngược = `total - clockwise`.
- Nếu `s == t`: `clockwise = 0`, đáp án `min(0, total) = 0` ✔.
- Hoán đổi không đổi đáp án vì khoảng cách đối xứng.

## 11. Độ phức tạp

Gọi `n = len(distance)`.

| Cách | Time | Space |
|---|---|---|
| Mô phỏng hai chiều | `O(n)` | `O(1)` |
| Tổng đoạn + phần bù | `O(n)` | `O(n)` cho slice (có thể `O(1)` với bản một vòng) |
| Một vòng lặp | `O(n)` | `O(1)` |
| Prefix sum | `O(n)` xây + `O(1)` mỗi truy vấn | `O(n)` |

## 12. Bộ test: mẫu, biên, đối chiếu ngẫu nhiên

In [13]:
cases = [
    ([1, 2, 3, 4], 0, 1, 1),
    ([1, 2, 3, 4], 0, 2, 3),
    ([1, 2, 3, 4], 0, 3, 4),
    ([1, 2, 3, 4], 3, 0, 4),                 # start > destination
    ([7, 10, 1, 12, 11, 14, 5, 0], 7, 2, 17),
    ([5], 0, 0, 0),                          # n = 1
    ([1, 2, 3], 1, 1, 0),                    # cùng trạm
    ([0, 0, 0, 0], 0, 2, 0),                 # khoảng cách 0
    ([3, 6, 7, 2, 9, 10, 7, 16, 11], 6, 2, 28),
]
solvers = [bus_simulate, bus_distance, bus_one_pass,
           Solution().distanceBetweenBusStops, SolutionOnePass().distanceBetweenBusStops,
           lambda d, s, t: BusRoute(d).query(s, t)]
for d, s, t, expected in cases:
    for f in solvers:
        assert f(d, s, t) == expected, (f, d, s, t)
    print(f"{str(d):<30} {s}->{t}: {expected} PASS")

[1, 2, 3, 4]                   0->1: 1 PASS
[1, 2, 3, 4]                   0->2: 3 PASS
[1, 2, 3, 4]                   0->3: 4 PASS
[1, 2, 3, 4]                   3->0: 4 PASS
[7, 10, 1, 12, 11, 14, 5, 0]   7->2: 17 PASS
[5]                            0->0: 0 PASS
[1, 2, 3]                      1->1: 0 PASS
[0, 0, 0, 0]                   0->2: 0 PASS
[3, 6, 7, 2, 9, 10, 7, 16, 11] 6->2: 28 PASS


`lambda d, s, t: ...` tạo một **hàm vô danh** ngắn — ở đây bọc lớp `BusRoute` thành hàm cùng chữ ký để đưa vào danh sách kiểm thử.

In [14]:
import random
random.seed(1184)
for _ in range(2000):
    n = random.randint(1, 10)
    d = [random.randint(0, 10) for _ in range(n)]
    s, t = random.randrange(n), random.randrange(n)
    expected = bus_simulate(d, s, t)
    for f in solvers[1:]:
        assert f(d, s, t) == expected
    assert bus_distance(d, s, t) == bus_distance(d, t, s)     # đối xứng
print("2000 test ngẫu nhiên khớp mô phỏng, và đáp án đối xứng. PASS")

2000 test ngẫu nhiên khớp mô phỏng, và đáp án đối xứng. PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Không hoán đổi khi `start > destination`
`sum(distance[3:0])` là đoạn **rỗng** = 0 → trả 0 sai.

### Lỗi 2: Cộng nhầm `distance[cur]` khi đi ngược chiều (phải là `distance[prev]`).

### Lỗi 3: Lệch một: `sum(distance[start:destination + 1])` — cộng thừa cạnh sau `destination`.

### Lỗi 4: Chỉ tính một chiều, quên chiều ngược.

In [15]:
def wrong_no_swap(d, s, t):
    cw = sum(d[s:t])
    return min(cw, sum(d) - cw)

def wrong_ccw_edge(d, s, t):
    n = len(d)
    cw, cur = 0, s
    while cur != t:
        cw += d[cur]; cur = (cur + 1) % n
    ccw, cur = 0, s
    while cur != t:
        ccw += d[cur]            # sai: phải là d[(cur - 1) % n]
        cur = (cur - 1) % n
    return min(cw, ccw)

def wrong_off_by_one(d, s, t):
    s, t = min(s, t), max(s, t)
    cw = sum(d[s:t + 1])
    return min(cw, sum(d) - cw)

args = ([1, 2, 3, 4], 3, 0)
print("đúng:", Solution().distanceBetweenBusStops(*args), "| không hoán đổi:", wrong_no_swap(*args))
args2 = ([1, 2, 3, 4], 0, 3)
print("đúng:", Solution().distanceBetweenBusStops(*args2), "| cạnh ngược sai:", wrong_ccw_edge(*args2))
args3 = ([1, 2, 3, 4], 0, 1)
print("đúng:", Solution().distanceBetweenBusStops(*args3), "| lệch một:", wrong_off_by_one(*args3))

đúng: 4 | không hoán đổi: 0
đúng: 4 | cạnh ngược sai: 1
đúng: 1 | lệch một: 3


## 14. Mẹo nhận dạng pattern

- **Vòng tròn** + hai chiều → đáp án thường là `min(x, total - x)`.
- Đi vòng trên mảng → dùng `% n`; hoặc "nhân đôi mảng" (`a + a`) để biến vòng thành đoạn thẳng.
- Khoảng cách giữa hai vị trí trên đường thẳng = tổng đoạn → **prefix sum** khi có nhiều truy vấn.
- Đổi vai trò cho `start <= destination` để không phải xử lý hai trường hợp.

Bài cùng họ: 1165 (Single-Row Keyboard), 134 (Gas Station), 2134 (Min Swaps to Group All 1's Together II), 918 (Maximum Sum Circular Subarray).

## 15. Biến thể

### 15.1. Chỉ được đi một chiều (thuận)
Không có `min`; nếu `start > destination` phải đi vòng qua trạm 0: `total - sum(distance[destination:start])`.

In [16]:
def clockwise_only(distance: List[int], start: int, destination: int) -> int:
    if start <= destination:
        return sum(distance[start:destination])
    return sum(distance) - sum(distance[destination:start])

print(clockwise_only([1, 2, 3, 4], 0, 3), clockwise_only([1, 2, 3, 4], 3, 0), clockwise_only([1, 2, 3, 4], 3, 1))

6 4 5


### 15.2. Kỹ thuật nhân đôi mảng
Nối `distance + distance` thì mọi đường thuận từ `s` tới `t` (kể cả vòng qua 0) là đoạn liên tiếp `[s, t + n)` khi `t < s`.

In [17]:
def clockwise_doubled(distance: List[int], start: int, destination: int) -> int:
    n = len(distance)
    d2 = distance + distance
    end = destination if destination >= start else destination + n
    return sum(d2[start:end])

for s in range(4):
    for t in range(4):
        assert clockwise_doubled([1, 2, 3, 4], s, t) == clockwise_only([1, 2, 3, 4], s, t)
print("nhân đôi mảng khớp công thức phần bù. PASS")

nhân đôi mảng khớp công thức phần bù. PASS


### 15.3. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Mô phỏng | đi từng trạm hai chiều | `O(n)` | `O(1)` | dễ hình dung, dễ lệch cạnh |
| Tổng đoạn + phần bù | `min(cw, total - cw)` | `O(n)` | `O(n)` slice | **nộp bài**, ngắn nhất |
| Một vòng | cộng có điều kiện | `O(n)` | `O(1)` | không tạo list |
| Prefix sum | `pre[t] - pre[s]` | `O(1)`/truy vấn | `O(n)` | nhiều truy vấn |

## 16. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
`distance = [3, 6, 7, 2, 9, 10, 7, 16, 11]`, `start = 6`, `destination = 2`. Tính chiều thuận, chiều ngược và đáp án.

### Bài 2 — Hoàn thiện lời giải chính

In [18]:
def bus_exercise(distance: List[int], start: int, destination: int) -> int:
    # TODO 1: hoán đổi nếu start > destination
    # TODO 2: clockwise = sum(...)
    # TODO 3: return min(...)
    pass

# assert bus_exercise([1, 2, 3, 4], 0, 3) == 4
# assert bus_exercise([1, 2, 3, 4], 3, 0) == 4

### Bài 3 — Mô phỏng chiều ngược
Viết hàm chỉ tính quãng đường **ngược chiều kim đồng hồ** bằng vòng `while`.

In [19]:
def counter_clockwise_exercise(distance: List[int], start: int, destination: int) -> int:
    # TODO: prev = (cur - 1) % n; cộng distance[prev]
    pass

# assert counter_clockwise_exercise([1, 2, 3, 4], 0, 1) == 9
# assert counter_clockwise_exercise([1, 2, 3, 4], 0, 3) == 4

### Bài 4 — Trạm xa nhất
Từ trạm `start`, tìm khoảng cách ngắn nhất **lớn nhất** tới một trạm khác (trạm "xa nhất" theo đường ngắn nhất).

In [20]:
def farthest_exercise(distance: List[int], start: int) -> int:
    # TODO: thử mọi destination, lấy max của đáp án bài chính
    pass

# assert farthest_exercise([1, 2, 3, 4], 0) == 4

### Bài 5 — Nhiều truy vấn với prefix sum

In [21]:
def many_queries_exercise(distance: List[int], queries: List[tuple]) -> List[int]:
    # TODO: xây pre một lần, mỗi truy vấn O(1)
    pass

# assert many_queries_exercise([1, 2, 3, 4], [(0, 1), (0, 2), (3, 0), (2, 2)]) == [1, 3, 4, 0]

### Bài 6 — Mini challenge: hành trình nhiều điểm
Xe phải lần lượt ghé các trạm trong list `stops` (bắt đầu từ `stops[0]`), mỗi chặng chọn chiều ngắn hơn. Tổng quãng đường?

In [22]:
def trip_exercise(distance: List[int], stops: List[int]) -> int:
    # TODO: cộng đáp án bài chính cho từng cặp liên tiếp
    pass

# assert trip_exercise([1, 2, 3, 4], [0, 2, 3, 1]) == 3 + 3 + 5

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | hoán đổi thành 2 → 6 | cạnh index 2..5 | `7+2+9+10 = 28`, tổng 71 → `min(28, 43)` |
| 2 | xem mục 5 | `start, destination = destination, start` | `min(cw, sum(distance) - cw)` |
| 3 | đi lùi | `(cur - 1) % n` | `ccw += distance[prev]; cur = prev` |
| 4 | duyệt mọi trạm | gọi lại hàm bài 2 | `max(bus(...) for d in range(n))` |
| 5 | `pre[0] = 0` | `cw = pre[t] - pre[s]` | `min(cw, total - cw)` |
| 6 | `zip(stops, stops[1:])` | tổng các chặng | `sum(bus(d, a, b) for a, b in ...)` |

## 18. Đáp án tham khảo

**Bài 1:** hoán đổi → `s = 2, t = 6`; thuận = `7 + 2 + 9 + 10 = 28`; tổng = 71; ngược = 43 → đáp án **28**.

In [23]:
def bus_answer(distance: List[int], start: int, destination: int) -> int:
    if start > destination:
        start, destination = destination, start
    cw = sum(distance[start:destination])
    return min(cw, sum(distance) - cw)

def counter_clockwise_answer(distance: List[int], start: int, destination: int) -> int:
    n = len(distance)
    ccw, cur = 0, start
    while cur != destination:
        prev = (cur - 1) % n
        ccw += distance[prev]
        cur = prev
    return ccw

def farthest_answer(distance: List[int], start: int) -> int:
    return max(bus_answer(distance, start, d) for d in range(len(distance)))

def many_queries_answer(distance: List[int], queries: List[tuple]) -> List[int]:
    pre = [0]
    for x in distance:
        pre.append(pre[-1] + x)
    total = pre[-1]
    res = []
    for s, t in queries:
        s, t = min(s, t), max(s, t)
        cw = pre[t] - pre[s]
        res.append(min(cw, total - cw))
    return res

def trip_answer(distance: List[int], stops: List[int]) -> int:
    return sum(bus_answer(distance, a, b) for a, b in zip(stops, stops[1:]))

assert Solution().distanceBetweenBusStops([3, 6, 7, 2, 9, 10, 7, 16, 11], 6, 2) == 28
assert bus_answer([1, 2, 3, 4], 0, 3) == 4
assert bus_answer([1, 2, 3, 4], 3, 0) == 4
assert counter_clockwise_answer([1, 2, 3, 4], 0, 1) == 9
assert counter_clockwise_answer([1, 2, 3, 4], 0, 3) == 4
assert farthest_answer([1, 2, 3, 4], 0) == 4
assert many_queries_answer([1, 2, 3, 4], [(0, 1), (0, 2), (3, 0), (2, 2)]) == [1, 3, 4, 0]
assert trip_answer([1, 2, 3, 4], [0, 2, 3, 1]) == 11
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def distanceBetweenBusStops(self, distance, start, destination):
        if start > destination:
            start, destination = destination, start
        cw = sum(distance[start:destination])
        return min(cw, sum(distance) - cw)
```

**Checklist:**
- [ ] Đã hoán đổi để `start <= destination`?
- [ ] Đoạn thuận là `distance[start:destination]` (không `+ 1`)?
- [ ] Chiều ngược = `tổng − thuận`?
- [ ] Test `start == destination`, `n = 1`, `start > destination`, khoảng cách bằng 0?